# Variables and Types

## Type Inference and Lowering

Numba CUDA programs are written in a subset of Python that is compiled to CUDA PTX code at runtime.

During the compilation process, the Numba CUDA compiler uses type inference and type lowering to deduce a concrete type for each variable for all points in a function. These deduced types are then used to generate the CUDA PTX code. These types include `int64`, `float32`, etc.

There is a requirement that the variable will have a single fixed type at each instruction in the kernel where that variable is live. If the Numba CUDA compiler is unable to determine the correct lowered type of a variable at all points, an exception will be raised and the kernel will not be compiled.

## NumPy Arrays

NumPy arrays are supported in Numba CUDA programs. Details on NumPy integration with Numba can be found in the [Numba documentation](https://numba.readthedocs.io/en/stable/).

The following is a Numba CUDA program which takes two vectors (NumPy arrays) `a` and `b`, adds them together and returns the result in `c`. This `vector_addition()` kernel is configured for full debugging information.

You can run this cell to see the results of the vector addition.

In [ ]:
from numba import cuda
import numpy as np

# Simple parallel vector addition kernel that runs on the GPU
# Full debugging support with line number, variable, and type information
# Optimizations are disabled
@cuda.jit(debug=True, opt=False)
def vector_addition(a, b, c):
    idx = cuda.grid(1)
    if idx < c.size:
        c[idx] = a[idx] + b[idx]

# The following code executes in the host Python interpreter
num_entries = 16
a = cuda.to_device(np.array(range(num_entries), dtype=np.int64))
b = cuda.to_device(np.array(range(num_entries), dtype=np.int64))
c = cuda.to_device(np.zeros_like(a))

# JIT the kernel at runtime and launch it on the GPU
vector_addition[1, num_entries](a, b, c)

# Print the inputs and output of the kernel
print(f"a: {a.copy_to_host()}")
print(f"b: {b.copy_to_host()}")
print(f"c: {c.copy_to_host()}")

print("All Done!")


## Polymorphic Variables

Numba CUDA supports a limited form of polymorphic variables.

In Python, variables are objects that are inherently polymorphic in nature,
meaning each assignment can change the variable's type in addition to its value.
This is in contrast to statically-typed languages like C / C++ where the type of a variable is declared at compile time and does not change.

For example, a Python variable can be assigned a boolean value, then an integer value, then a float value, and so on. Whereas a C++ variable of type `int` is always an `int` and can only hold integral values.

Run the following cell to see how the variable `x` changes both its value and type during the execution of the kernel.

In [ ]:
from numba import cuda
import numpy as np

@cuda.jit(debug=True, opt=False)
def polymorphic_example():
    x = 1234567890                              # Type is int64
    print("x =", x, "(type: int64)")
    x = True                                    # Type is bool
    print("x =", x, "(type: bool)")
    x = 3.141592653589793                       # Type is float64
    print("x =", x, "(type: float64)")
    print("All Done!")

# The following code is executed in the host Python interpreter
polymorphic_example[1, 1]()

Note that some variables may be polymorphic in nature for non-obvious reasons. Consider the following example where `input` is a `uint32` on entry, so at the first assignment `x` takes on that type. The next assignment of `x = 1234567890` changes the type of `x` from an unsigned type to a signed type (as Python literal integers are signed). This makes `x` polymorphic in nature even though it's not immediately obvious that it can take on different types.

In [ ]:
from numba import cuda
import numpy as np

# Provide a type signature to the compiler to indicate that the argument
# "input" is passed in as a 32-bit unsigned integer.
@cuda.jit("void(uint32)", debug=True, opt=False)
def polymorphic_example(input):
    x = input                                   # Type is uint32
    print("x =", x, "(type: uint32)")
    x = 1234567890                              # Type is now int64
    print("x =", x, "(type: int64)")

# JIT and launch the kernel
polymorphic_example[1, 1](1234)

The debugger has support for Numba CUDA polymorphic variables. When the debugger reads or writes a polymorphic variable in Numba CUDA, it determines the type of the variable for the current point in the kernel where the debugger is stopped. If the variable is a float, it will print as a floating point number, and inspecting the type of the variable will report that it's a float. The next line of code could assign an integer value to the variable, which will change both the value and the type of the variable **at that point in the kernel**.